# ⚖️ Niti Liability LLM v2: Legal Syllogism & Precision Fast-Path
### ระบบวินิจฉัยความรับผิดทางอาญาจากข้อเท็จจริงคดีศาลฎีกาไทย (TSCC Benchmark)
**สถาปัตยกรรมระดับ Production (Zero Fine-Tuning):**
* **Hybrid Retrieval (70:30)**: ChromaDB (BGE-M3 Dense Vector) + BM25 Lexical Keyword
* **Precision Reranking**: BAAI Cross-Encoder Reranker (`bge-reranker-base`)
* **Legal Syllogism Reasoning**: บังคับคิดตามตรรกวิทยาทางนิติศาสตร์ (Major Premise ➔ Minor Premise ➔ Conclusion)
* **Domain Post-Processing**: Twin-Article Decision Gate (334 vs 352, 326 vs 328) + Anti-Dangling Rule
* **Fast Parallel Companion Verification**: ตรวจจับมาตราคู่หูแบบคู่ขนานระดับมิลลิวินาที (Throughput ~8 it/s)
* **Resilient Infrastructure**: Multi-threading 25 Workers + 3-Stage Auto-Recovery + Real-time Checkpoints


In [1]:
# Setup Environment
!pip install -q --no-cache-dir --upgrade --target /usr/local/lib/python3.13/dist-packages "huggingface_hub>=1.29.0" "tokenizers>=0.23.1"
!pip install -q chromadb rank_bm25 pythainlp openai pydantic datasets pandas openpyxl tqdm

import os, re, glob, json, time, copy, random
from datetime import datetime
from collections import defaultdict
from itertools import combinations
import concurrent.futures

import pandas as pd
import numpy as np
from tqdm import tqdm
from datasets import load_dataset
from pythainlp.tokenize import word_tokenize
from pydantic import BaseModel, Field, field_validator

tqdm.pandas()
print("✅ [Step 1] Library พร้อมใช้งานสมบูรณ์ (HF 1.29.0, ST 5.7.0, Chroma 1.5.9)")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 253.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.9/839.9 kB 260.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 199.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 273.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.5/162.5 kB 284.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.7/221.7 kB 336.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 265.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 282.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.1/85.1 kB 267.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.0/130.0 kB 330.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.6/801.6 kB 331.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 304.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 📦 Phase 1: การเตรียมชุดข้อมูล (TSCC Dataset Loading & Preprocessing)
โหลดข้อมูลคำพิพากษาศาลฎีกา (Judgements) และประมวลกฎหมายอาญา (Master Penal Code) จากคลังข้อมูล TSCC
พร้อมแปลงรูปแบบรหัสกฎหมาย (Data Normalization: CC-xxx ➔ เลขมาตรา) และตัดข้อเท็จจริงที่ซ้ำซ้อนออก


In [2]:
# 2.1 TSCC Dataset
if not os.path.exists("tscc-dataset"):
    os.system("git clone https://github.com/KevinMercury/tscc-dataset.git")
else:
    print("📁 tscc-dataset folder already exists.")

# 2.2 ค้นหา Path CSV
all_csvs = glob.glob("tscc-dataset/**/*.csv", recursive=True)
path_map = {}
for f in all_csvs:
    fname = os.path.basename(f).lower()
    if "judgement" in fname: path_map["judgement"] = f
    if "law" in fname: path_map["law"] = f

if "judgement" not in path_map or "law" not in path_map:
    raise FileNotFoundError(f"❌ ไม่พบไฟล์! พบ: {all_csvs}")

# 2.3 TSCC Judgement
df_judgement = pd.read_csv(path_map["judgement"])
initial_count = len(df_judgement)
df_judgement = df_judgement.drop_duplicates(subset=["fact"]).reset_index(drop=True)
print(f"🗑️  ลบซ้ำ: {initial_count - len(df_judgement)} รายการ → เหลือ {len(df_judgement)} คดี")

def clean_lawids(text: str) -> list:
    """แปลง lawids string → list เลขมาตรา"""
    if pd.isna(text): return []
    sections = re.findall(r"CC-(\d+)", text)
    return list(set([str(int(s)) for s in sections]))

df_judgement["clean_sections"] = df_judgement["lawids"].apply(clean_lawids)

# 2.4 TSCC Law → Master List
df_law = pd.read_csv(path_map["law"])
section_col = "lawsection" if "lawsection" in df_law.columns else df_law.columns[2]
content_col = "content" if "content" in df_law.columns else df_law.columns[-1]
master_law_dict = dict(zip(
    df_law[section_col].astype(str).apply(lambda x: str(int(x)) if x.isdigit() else x),
    df_law[content_col],
))
list_of_all_legal_sections = set(master_law_dict.keys())

print(f"\n✅ [Step 2] โหลดข้อมูลสำเร็จ")
print(f"   TSCC Judgement : {len(df_judgement)} คดี")
print(f"   มาตราในระบบ   : {len(list_of_all_legal_sections)} มาตรา")


Cloning into 'tscc-dataset'...


🗑️  ลบซ้ำ: 161 รายการ → เหลือ 1046 คดี

✅ [Step 2] โหลดข้อมูลสำเร็จ
   TSCC Judgement : 1046 คดี
   มาตราในระบบ   : 76 มาตรา


## 🧠 Phase 2: คลังจัดเก็บเวกเตอร์กฎหมาย (ChromaDB Vector Store & BGE-M3)
สร้างคอลเลกชันประมวลกฎหมายอาญาใน ChromaDB โดยฝังเวกเตอร์ (Dense Embedding) ด้วยโมเดล `BAAI/bge-m3` (1024-dim)
เพื่อรองรับการค้นหาตาม "ความหมาย เจตนา และบริบททางนิติศาสตร์"


In [3]:
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = "intfloat/multilingual-e5-large"
CHROMA_PATH = "/kaggle/working/chroma_db"
BATCH_SIZE = 100

print(f"⏳ โหลด Embedding Model: {EMBED_MODEL_NAME}")
embed_model = SentenceTransformer(EMBED_MODEL_NAME)
print(f"✅ Embedding Model พร้อม | Dim: {embed_model.get_sentence_embedding_dimension()}")

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH, settings=Settings(anonymized_telemetry=False))

# Primary: TSCC Law Collection
law_collection = chroma_client.get_or_create_collection(name="tscc_law_primary", metadata={"hnsw:space": "cosine"})

if law_collection.count() == 0:
    ids, documents, metadatas, embeddings = [], [], [], []
    for section_num, content in tqdm(master_law_dict.items(), desc="Indexing TSCC Law"):
        text = f"มาตรา {section_num}: {content}"
        emb = embed_model.encode(f"passage: {text}", normalize_embeddings=True).tolist()
        ids.append(f"law_{section_num}")
        documents.append(text)
        metadatas.append({"section": section_num, "source": "tscc_law"})
        embeddings.append(emb)
    for i in range(0, len(ids), BATCH_SIZE):
        law_collection.upsert(ids=ids[i:i+BATCH_SIZE], documents=documents[i:i+BATCH_SIZE],
                              metadatas=metadatas[i:i+BATCH_SIZE], embeddings=embeddings[i:i+BATCH_SIZE])
    print(f"✅ Index TSCC Law: {law_collection.count()} มาตรา")
else:
    print(f"⚠️  tscc_law_primary มีข้อมูลอยู่แล้ว {law_collection.count()} รายการ")

print(f"\n✅ [Step 3] ChromaDB พร้อม | Path: {CHROMA_PATH}")


⏳ โหลด Embedding Model: intfloat/multilingual-e5-large


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

/tmp/ipykernel_23/3358444206.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"✅ Embedding Model พร้อม | Dim: {embed_model.get_sentence_embedding_dimension()}")


✅ Embedding Model พร้อม | Dim: 1024


Indexing TSCC Law: 100%|██████████| 76/76 [00:02<00:00, 26.72it/s]

✅ Index TSCC Law: 76 มาตรา

✅ [Step 3] ChromaDB พร้อม | Path: /kaggle/working/chroma_db


## 📚 Phase 3: คัดเลือกตัวอย่างสาธิต (Diverse Few-Shot Selection)
สุ่มเลือกตัวอย่างคำพิพากษาที่หลากหลายหมวดความผิด (ชีวิต ร่างกาย ทรัพย์ เสรีภาพ เอกสาร) จาก Training Set 
เพื่อใช้เป็น Few-shot Demonstrations นำทางให้ LLM ตอบถูกรูปแบบและเข้าใจแนวทางการวินิจฉัยของศาลฎีกา


In [4]:
random.seed(42)
DEV_SIZE  = 81
TEST_SIZE = 965

all_indices = list(range(len(df_judgement)))
random.shuffle(all_indices)
dev_indices  = all_indices[:DEV_SIZE]
test_indices = all_indices[DEV_SIZE : DEV_SIZE + TEST_SIZE]

df_dev  = df_judgement.iloc[dev_indices].reset_index(drop=True)
df_test = df_judgement.iloc[test_indices].reset_index(drop=True)

print(f"✅ แบ่ง Dataset:")
print(f"   Dev Set  : {len(df_dev)} คดี")
print(f"   Test Set : {len(df_test)} คดี")

# Pool สำหรับ Few-shot (ไม่ overlap)
all_used_indices = set(dev_indices) | set(test_indices)
pool = df_judgement.loc[~df_judgement.index.isin(all_used_indices)].copy()
if pool.empty:
    print("⚠️ Pool ว่าง! สุ่มจากต้นฉบับ")
    pool = df_judgement.sample(min(20, len(df_judgement)), random_state=42).copy()
print(f"📊 Pool few-shot: {len(pool)} คดี")

def get_examples_by_sections(df, must_have, nice_to_have=None, n=2):
    if df.empty: return pd.DataFrame()
    def has_all(secs, targets):
        return all(t in secs for t in targets) if isinstance(secs, list) else False
    matched = df[df['clean_sections'].apply(lambda s: has_all(s, must_have))].copy()
    if not matched.empty:
        if nice_to_have:
            matched['has_nice'] = matched['clean_sections'].apply(
                lambda s: any(t in s for t in nice_to_have) if isinstance(s, list) else False)
            matched = matched.sort_values('has_nice', ascending=False)
        matched['fact_len'] = matched['fact'].str.len()
        matched = matched[(matched['fact_len'] >= 100) & (matched['fact_len'] <= 800)]
    return matched.head(n)

ex_a = get_examples_by_sections(pool, must_have=['288'], nice_to_have=['68','83'], n=2)
ex_b_mask = pool['clean_sections'].apply(lambda s: any(x in s for x in ['334','335']) if isinstance(s, list) else False)
ex_b = pool[ex_b_mask].copy()
if not ex_b.empty:
    ex_b['fact_len'] = ex_b['fact'].str.len()
    ex_b = ex_b[(ex_b['fact_len'] >= 100) & (ex_b['fact_len'] <= 800)].head(1)
ex_c = get_examples_by_sections(pool, must_have=['83'], nice_to_have=['288','334'], n=1)

few_shot_df = pd.concat([ex_a, ex_b, ex_c]).drop_duplicates(subset=['fact']).reset_index(drop=True)

def build_few_shot_string(df: pd.DataFrame) -> str:
    if df.empty: return 'ไม่มีตัวอย่าง'
    blocks = []
    for i, row in df.iterrows():
        secs = row.get('clean_sections', [])
        secs_str = ', '.join(f'ม.{s}' for s in secs) if isinstance(secs, list) else str(secs)
        blocks.append(f'### ตัวอย่างที่ {i+1}\nข้อเท็จจริง: {row["fact"]}\nมาตราที่เกี่ยวข้อง: {secs_str}\n')
    return '\n'.join(blocks)

FEW_SHOT_STRING = build_few_shot_string(few_shot_df)
print(f"✅ [Step 4] Few-shot พร้อมใช้งาน ({len(few_shot_df)} ตัวอย่าง)")

✅ แบ่ง Dataset:
   Dev Set  : 81 คดี
   Test Set : 965 คดี
⚠️ Pool ว่าง! สุ่มจากต้นฉบับ
📊 Pool few-shot: 20 คดี
✅ [Step 4] Few-shot พร้อมใช้งาน (4 ตัวอย่าง)


## 🔍 Phase 4: ระบบค้นหาไฮบริดและจัดอันดับซ้ำ (Hybrid Retrieval & Cross-Encoder)
ผสานการทำงานระหว่าง (Hybrid Integration) **BM25 Lexical Search** (คำเฉพาะทาง/เลขมาตรา) และ **Vector Search** ในอัตราส่วน 30:70
พร้อมต่อยอดด้วย **Cross-Encoder Reranker (`bge-reranker-base`)** เพื่อวิเคราะห์ความสัมพันธ์แบบคู่ (Cross-Attention) 
และใช้ **Cluster Expansion (±3 มาตรา)** ป้องกันการตกหล่นของมาตราย่อย


In [5]:
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

_law_texts    = [f'มาตรา {s}: {c}' for s, c in master_law_dict.items()]
_law_sections = list(master_law_dict.keys())
_tokenized    = [word_tokenize(t, engine='newmm') for t in _law_texts]
bm25_index    = BM25Okapi(_tokenized)

# Reranker Model (Lightweight Cross-Encoder for precision)
print("⏳ โหลด Cross-Encoder Reranker...")
try:
    reranker_model = CrossEncoder("BAAI/bge-reranker-base")
    print("✅ Cross-Encoder Reranker พร้อมใช้งาน")
except Exception as e:
    print(f"⚠️ Reranker fallback to score sorting: {e}")
    reranker_model = None


def hybrid_retrieve(query: str, top_k: int = 15, bm25_weight: float = 0.3, vector_weight: float = 0.7, rerank_top_n: int = 5) -> dict:
    """Hybrid Retrieval (Vector + BM25) + Cross-Encoder Reranker to Top-N"""
    query_emb = embed_model.encode(f'query: {query}', normalize_embeddings=True).tolist()
    vector_results = law_collection.query(
        query_embeddings=[query_emb],
        n_results=min(top_k * 2, law_collection.count()),
        include=['documents', 'metadatas', 'distances'],
    )
    tokenized_query = word_tokenize(query, engine='newmm')
    bm25_scores     = bm25_index.get_scores(tokenized_query)
    top_bm25_idx    = np.argsort(bm25_scores)[::-1][:top_k * 2]

    combined = {}
    for doc, meta, dist in zip(vector_results['documents'][0], vector_results['metadatas'][0], vector_results['distances'][0]):
        sec = meta.get('section', '')
        combined[sec] = combined.get(sec, 0) + vector_weight * (1 - dist)
    max_bm25 = bm25_scores[top_bm25_idx[0]] if len(top_bm25_idx) > 0 else 1e-9
    for idx in top_bm25_idx:
        sec = _law_sections[idx]
        combined[sec] = combined.get(sec, 0) + bm25_weight * (bm25_scores[idx] / (max_bm25 + 1e-9))

    candidate_secs = sorted(combined, key=lambda x: -combined[x])[:top_k]
    
    # Reranking stage if CrossEncoder available
    if reranker_model is not None and len(candidate_secs) > rerank_top_n:
        pairs = [(query, f"มาตรา {s}: {master_law_dict.get(s, '')}") for s in candidate_secs]
        rerank_scores = reranker_model.predict(pairs)
        ranked_indices = np.argsort(rerank_scores)[::-1][:rerank_top_n]
        top_secs = [candidate_secs[i] for i in ranked_indices]
    else:
        top_secs = candidate_secs[:rerank_top_n]

    results, context_parts = [], []
    for sec in top_secs:
        content  = master_law_dict.get(sec, '')
        doc_text = f'มาตรา {sec}: {content}'
        results.append({'id': sec, 'document': doc_text, 'score': combined.get(sec, 1.0)})
        context_parts.append(doc_text)

    # Neighbor Expansion +-1 for top 3
    neighbors = set()
    for sec in top_secs[:3]:
        try:
            n = int(re.sub(r'\D', '', sec))
            for nb in [str(n-1), str(n+1)]:
                if nb in master_law_dict and nb not in top_secs:
                    neighbors.add(nb)
        except ValueError:
            pass
    for nb in neighbors:
        content  = master_law_dict.get(nb, '')
        doc_text = f'มาตรา {nb}: {content}'
        results.append({'id': nb, 'document': doc_text, 'score': 0.0})
        context_parts.append(doc_text)

    return {'results': results, 'context_text': '\n\n'.join(context_parts), 'context_section_ids': [r['id'] for r in results]}

print("✅ [Step 5] hybrid_retrieve() + Reranker พร้อม")


⏳ โหลด Cross-Encoder Reranker...


config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

✅ Cross-Encoder Reranker พร้อมใช้งาน
✅ [Step 5] hybrid_retrieve() + Reranker พร้อม


## ⚖️ Phase 5: พรอมต์นิติศาสตร์และกฎอนุรักษ์นิยม (Legal Syllogism & Judicial Rules)
ออกแบบ System Prompt บังคับให้ LLM ดำเนินกระบวนการคิดตาม **Legal Syllogism (ตรรกวิทยาทางนิติศาสตร์)**
พร้อมกำหนด **[CONSERVATIVE JUDICIAL RULES]** ห้ามเดาข้อหาแถม ป้องกันการตั้งข้อหาเกินจริง (Over-charging) เพื่อรักษา Precision สูงสุด


In [6]:
SYSTEM_PROMPT = """คุณคือผู้ช่วยผู้พิพากษาศาลอาญา ทำหน้าที่วินิจฉัยฐานความผิดตามหลัก Legal Syllogism (ตรรกวิทยาทางนิติศาสตร์)

กระบวนการคิด 3 ขั้นตอน:
1. MAJOR PREMISE (ตัวบทกฎหมาย): ดึงเฉพาะองค์ประกอบความผิดของมาตราใน Reference
2. MINOR PREMISE (ปรับข้อเท็จจริงเข้ากับตัวบท): เทียบพฤติการณ์จำเลยทีละองค์ประกอบ
3. CONCLUSION (บทสรุป): ฟันธงเลขมาตราเฉพาะข้อที่เข้าครบทุกองค์ประกอบ

[CONSERVATIVE JUDICIAL RULES - ห้ามตัดสินเกินข้อเท็จจริง]:
- ตัดสินเฉพาะ 'เจตนาหลักแห่งคดี' ห้ามคาดเดาเจตนาซ้อน หรือแตกกรรมความผิดโดยไม่มีพฤติการณ์แยกชัดเจน
- หากข้อเท็จจริงระบุว่า 'ลักเอาทรัพย์ไป' แม้มีเหตุการณ์กระทบกระทั่งแต่ไม่ได้ใช้กำลังเพื่อเอาทรัพย์โดยตรง ให้คงไว้ที่ความผิดฐานลักทรัพย์ (ม.334/335) ห้ามกระโดดไปชิงทรัพย์ (ม.339) หรือพยายามฆ่า (ม.288/80)
- บาดเจ็บทั่วไป = ม.295 | บาดเจ็บสาหัส (รักษาเกิน 20 วัน/พิการ/ทุพพลภาพ) = ม.297 | ประมาทตาย = ม.291 | ทำร้ายแต่ตาย = ม.290

[ANCHOR RULES & CRITICAL TESTS]:
1. PROPERTY (334 vs 352):
   - มีอำนาจครอบครองอยู่แล้วแล้วเบียดบัง = ม.352 (ยักยอก)
   - ทรัพย์อยู่ในความครอบครองของผู้อื่นแล้วเอาไป = ม.334 (ลักทรัพย์)
   - ลักทรัพย์ + (กลางคืน / ร่วมกัน 2 คน / ในเคหสถาน) = ม.335 (เหตุฉกรรจ์) เสมอ
2. FRAUD vs THEFT (341 vs 334):
   - หลอกให้เจ้าของส่งมอบกรรมสิทธิ์/การครอบครอง = ม.341 (ฉ้อโกง)
   - หลอกเพื่อเบี่ยงเบนความสนใจแล้วแอบหยิบเอาไปเอง = ม.334 (ลักทรัพย์โดยใช้กลอุบาย)
3. DEFAMATION (326 vs 328):
   - โพสต์ Facebook, LINE กลุ่ม, สื่อออนไลน์ = ม.328 (โฆษณา) ห้ามตอบแค่ 326
4. ATTEMPT & ACCOMPLICE:
   - มีข้อความ "พยายาม/กระสุนด้าน/ไม่ตาย" = เพิ่ม ม.80 หรือ ม.81
   - มีข้อความ "ร่วมกัน/ด้วยกัน" = เพิ่ม ม.83

กฎเหล็ก:
- JSON format เท่านั้น
- sections ต้องเป็นตัวเลข string เช่น ["334", "83"]
- ห้ามระบุมาตราที่ไม่อยู่ใน Reference
"""

USER_PROMPT_TEMPLATE = """วินิจฉัยคดีตามหลัก Legal Syllogism:

=== ข้อเท็จจริง (Fact) ===
{fact}

=== ตัวบทกฎหมายอ้างอิง (Reference) ===
{legal_context}

ตอบในรูปแบบ JSON:
{{
  "case_id": "{case_id}",
  "major_premise": "สรุปองค์ประกอบความผิดของมาตราที่เกี่ยวข้อง",
  "minor_premise": "ปรับข้อเท็จจริงของจำเลยเข้ากับแต่ละองค์ประกอบ",
  "differential_analysis": "จุดตัดชี้ขาดที่เลือกมาตรานี้และตัดมาตราคู่แฝดออก",
  "criminal_charge": "ชื่อฐานความผิดทางการ",
  "sections": ["เลขมาตราหลัก", "มาตราสนับสนุน"],
  "confidence_score": 0-100,
  "final_reasoning": "สรุปหลักนิติศาสตร์ 1 ประโยค"
}}
"""

def build_prompt(fact: str, retrieved: dict, case_id: str = 'N/A') -> tuple:
    context_str = '\n\n'.join([f"[ID: {r['id']}] {r['document']}" for r in retrieved['results']])
    user = USER_PROMPT_TEMPLATE.format(fact=fact, legal_context=context_str, case_id=case_id)
    return SYSTEM_PROMPT, user

print("✅ [Step 6] build_prompt() พร้อม (Legal Syllogism + Conservative Judicial Rules)")


✅ [Step 6] build_prompt() พร้อม (Legal Syllogism + Conservative Judicial Rules)


## ⚙️ Phase 6: ท่อประมวลผลและการตัดสินใจเฉพาะทาง (Domain Decision Gates & Engine)
ติดตั้งระบบเชื่อมต่อ **OpenTyphoon 30B API** ควบคู่กับระบบคัดกรองทางนิติศาสตร์:
1. **Twin-Article Decision Gate**: แยกแยะข้อหาที่สับสนบ่อย (ลักทรัพย์ 334 vs ยักยอก 352, หมิ่นประมาท 326 vs ทางออนไลน์ 328)
2. **Anti-Dangling Rule**: ป้องกันการตอบมาตราประกอบ (ม.83 ตัวการ, ม.80 พยายาม) มาเดี่ยวๆ โดยไม่มีฐานความผิดหลัก


In [7]:
from openai import OpenAI
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
TYPHOON_API_KEY = secrets.get_secret("TYPHOON_API_KEY")

typhoon_client = OpenAI(
    api_key=TYPHOON_API_KEY,
    base_url="https://api.opentyphoon.ai/v1",
    timeout=120.0,
)

TYPHOON_MODEL = "typhoon-v2.5-30b-a3b-instruct"
MAX_TOKENS    = 2048
TEMPERATURE   = 0.1

def clean_section_format(section_list: list) -> list:
    if not section_list: return []
    cleaned = []
    for s in section_list:
        s_clean = str(s).replace('มาตรา', '').replace('ลำดับที่', '').strip()
        s_clean = s_clean.split('(')[0].strip()
        s_clean = re.sub(r'[^0-9/]', '', s_clean)
        if s_clean: cleaned.append(s_clean)
    return list(set(cleaned))

def call_typhoon(system_prompt: str, user_prompt: str, max_retries: int = 3) -> str | None:
    for attempt in range(max_retries):
        try:
            response = typhoon_client.chat.completions.create(
                model=TYPHOON_MODEL,
                messages=[
                    {'role': 'system', 'content': system_prompt},
                    {'role': 'user',   'content': user_prompt},
                ],
                max_tokens=MAX_TOKENS,
                temperature=TEMPERATURE,
                response_format={"type": "json_object"},
            )
            return response.choices[0].message.content
        except Exception as e:
            time.sleep(2 ** attempt)
    return None

def parse_llm_json(raw_text: str) -> dict | None:
    if not raw_text: return None
    try:
        return json.loads(raw_text)
    except Exception:
        cleaned = re.sub(r'`json\s*|\s*`', '', raw_text).strip()
        try: return json.loads(cleaned)
        except Exception: return None

# Fast Local Keyword Extractor (No LLM Network Overhead)
LEGAL_KEYWORDS_LIST = [
    "ลักทรัพย์", "ยักยอก", "ฉ้อโกง", "ชิงทรัพย์", "ปล้นทรัพย์", "ทำร้ายร่างกาย",
    "ฆ่า", "พยายาม", "ร่วมกัน", "โฆษณา", "หมิ่นประมาท", "บุกรุก", "เอกสารปลอม",
    "อาวุธ", "กลางคืน", "เคหสถาน", "สาหัส", "ประมาท", "เจ้าพนักงาน"
]

def fast_extract_legal_query(fact: str) -> str:
    matched = [kw for kw in LEGAL_KEYWORDS_LIST if kw in fact]
    return ", ".join(matched) if matched else fact[:150]

# Twin-Article Decision Gate (Resolving Zero-F1 Bottlenecks)
def twin_decision_gate(fact: str, pred_sections: list) -> list:
    has_theft = any(s in pred_sections for s in ["334", "335"])
    has_embezzle = "352" in pred_sections
    
    # 1. Check 334/335 vs 352
    if has_theft or has_embezzle:
        if any(w in fact for w in ["ฝากไว้", "มอบหมายให้ดูแล", "ครอบครองแทน", "ยืมไปแล้วไม่คืน", "เบียดบัง"]):
            pred_sections = [s for s in pred_sections if s not in ["334", "335"]]
            if "352" not in pred_sections: pred_sections.append("352")
        elif any(w in fact for w in ["แอบหยิบ", "ลอบเอาไป", "ล้วงกระเป๋า", "งัดแงะ", "หยิบไปโดยไม่บอก"]):
            pred_sections = [s for s in pred_sections if s != "352"]
            if not any(s in pred_sections for s in ["334", "335"]): pred_sections.append("334")

    # 2. Upgrade 334 -> 335 if aggravating circumstance found
    if "334" in pred_sections and "335" not in pred_sections:
        if any(w in fact for w in ["กลางคืน", "ร่วมกัน", "เคหสถาน", "ยานพาหนะ", "รั้ว", "ซ่อนหน้า"]):
            pred_sections.remove("334")
            pred_sections.append("335")

    # 3. Defamation 326 -> 328
    if "326" in pred_sections and "328" not in pred_sections:
        if any(w in fact for w in ["โพสต์", "facebook", "ไลน์", "line", "กระจายข่าว", "โฆษณา", "อินเทอร์เน็ต"]):
            pred_sections.remove("326")
            pred_sections.append("328")

    # 4. Bodily Harm vs Death: 297 vs 290
    if "297" in pred_sections and any(w in fact for w in ["ถึงแก่ความตาย", "ตาย", "เสียชีวิต"]) and not any(s in pred_sections for s in ["288", "289", "290", "291"]):
        pred_sections.append("290")

    return pred_sections

# Anti-Dangling Rule: ห้ามตอบ ม.83/80 โดดๆ โดยไม่มีมาตราหลัก
DEPENDENT_SECS_SET = {'83', '84', '86', '87', '80', '81', '68'}
def fix_dangling_sections(pred_sections: list, context_ids: list, fact: str) -> list:
    non_dep = [s for s in pred_sections if s not in DEPENDENT_SECS_SET]
    if not non_dep and pred_sections:
        # มีเฉพาะมาตราสนับสนุน เช่น ['83'] -> หามาตราหลักที่เหมาะสมที่สุดจาก context
        for candidate in context_ids:
            if candidate not in DEPENDENT_SECS_SET and candidate in master_law_dict:
                # ถ้ามีเรื่องทรัพย์
                if any(w in fact for w in ["ทรัพย์", "เงิน", "สร้อย", "รถ", "ของ"]) and candidate in ["334", "335", "352", "341"]:
                    pred_sections.append(candidate)
                    break
                # ถ้ามีเรื่องทำร้าย/ตาย
                elif any(w in fact for w in ["ตี", "แทง", "ยิง", "ฟัน", "ตาย", "เจ็บ"]) and candidate in ["288", "289", "290", "295", "297"]:
                    pred_sections.append(candidate)
                    break
        if not [s for s in pred_sections if s not in DEPENDENT_SECS_SET]:
            # Fallback default
            pred_sections.append("334" if "ทรัพย์" in fact else "295")
    return pred_sections

def post_process_llm_output(llm_output: dict, legal_db: set, context_section_ids: list, fact: str, anti_dangling: bool = True) -> dict:
    raw_sections = llm_output.get('sections', [])
    if isinstance(raw_sections, (str, int)): raw_sections = [str(raw_sections)]
    normalized   = clean_section_format(raw_sections)
    
    gated_sections = twin_decision_gate(fact, normalized)
    if anti_dangling:
        gated_sections = fix_dangling_sections(gated_sections, context_section_ids, fact)
        
    verified       = [s for s in gated_sections if s in legal_db]
    hallucinated   = [s for s in gated_sections if s not in legal_db]
    
    original_conf  = float(llm_output.get('confidence_score', 85))
    penalty        = len(hallucinated) * 20
    adjusted_conf  = max(0, min(100, original_conf - penalty))
    
    llm_output.update({
        'verified_sections': verified,
        'sections': verified,
        'adjusted_confidence': adjusted_conf,
        'verdict_mode': 'CONFIDENT' if adjusted_conf >= 80 else 'REVIEW_REQUIRED',
    })
    return llm_output

def analyze_criminal_liability_custom(fact: str, case_id: str = 'N/A', strat: dict = None) -> dict:
    if strat is None:
        strat = {"top_k": 25, "use_rerank": False, "anti_dangling": True}
        
    top_k = strat.get("top_k", 25)
    use_rerank = strat.get("use_rerank", False)
    rerank_n = strat.get("rerank_n", 20)
    anti_dang = strat.get("anti_dangling", True)
    expand = strat.get("expand", 1)
    
    keywords = fast_extract_legal_query(fact)
    
    if use_rerank:
        res_fact = hybrid_retrieve(fact, top_k=top_k, rerank_top_n=rerank_n)
        res_key  = hybrid_retrieve(keywords, top_k=top_k, rerank_top_n=rerank_n)
    else:
        # Full Legacy Hybrid (Top-25)
        res_fact = hybrid_retrieve(fact, top_k=top_k, rerank_top_n=top_k)
        res_key  = hybrid_retrieve(keywords, top_k=top_k, rerank_top_n=top_k)
        
    combined_ids = list(set(res_fact['context_section_ids']) | set(res_key['context_section_ids']))

    # Cluster Expansion (+-expand)
    if expand > 1:
        cluster_neighbors = set()
        for sec in list(combined_ids)[:5]:
            try:
                n = int(re.sub(r'\D', '', sec))
                for step in range(1, expand + 1):
                    for nb in [str(n - step), str(n + step)]:
                        if nb in master_law_dict and nb not in combined_ids:
                            cluster_neighbors.add(nb)
            except Exception:
                pass
        combined_ids.extend(list(cluster_neighbors))

    # Essentials Fallback
    if len(combined_ids) < 12:
        essentials = ["334", "335", "341", "352", "288", "289", "290", "291", "295", "297", "298", "83", "80"]
        combined_ids.extend([e for e in essentials if e not in combined_ids])

    # Force inject according to factual indicators
    if any(k in fact for k in ["ร่วมกัน", "ด้วยกัน"]) and "83" not in combined_ids:
        combined_ids.append("83")
    if any(k in fact for k in ["พยายาม", "กระสุนด้าน", "ไม่ตาย"]) and "80" not in combined_ids:
        combined_ids.append("80")

    context_str = "\n\n".join([f"[ID: {sid}] มาตรา {sid}: {master_law_dict[sid]}" for sid in combined_ids if sid in master_law_dict])
    user_p = USER_PROMPT_TEMPLATE.format(fact=fact, legal_context=context_str, case_id=case_id)

    raw = call_typhoon(SYSTEM_PROMPT, user_p)
    llm_dict = parse_llm_json(raw)

    if not llm_dict or not llm_dict.get("sections"):
        force_p = user_p + "\n\n**คำสั่งพิเศษ: ต้องเลือกมาตราอย่างน้อย 1 ข้อตามพฤติการณ์ ห้ามตอบลิสต์ว่าง**"
        raw_retry = call_typhoon(SYSTEM_PROMPT, force_p)
        llm_dict = parse_llm_json(raw_retry)

    if not llm_dict:
        return {'case_id': case_id, 'error': 'LLM Failure'}

    return post_process_llm_output(llm_dict, list_of_all_legal_sections, combined_ids, fact, anti_dangling=anti_dang)

print("✅ [Step 7] Universal Strategy Engine + Anti-Dangling Rule พร้อมใช้งาน")


✅ [Step 7] Universal Strategy Engine + Anti-Dangling Rule พร้อมใช้งาน


## 🛡️ Phase 7: การตรวจสอบโครงสร้างผลลัพธ์ (Pydantic Schema & Output Validator)
กำหนด Schema ด้วย **Pydantic (LLMOutput)** เพื่อการันตีความถูกต้องของข้อมูล (Data Contract)
และฟังก์ชันคำนวณคะแนนประเมินผลทางการ (Precision, Recall, F1-Score)


In [8]:
class LLMOutput(BaseModel):
    """Pydantic Schema สำหรับ Legal Syllogism Output"""
    case_id:               str
    major_premise:         str = ""
    minor_premise:         str = ""
    differential_analysis: str = ""
    criminal_charge:       str = ""
    sections:              list[str]
    confidence_score:      float = Field(default=85.0, ge=0, le=100)
    final_reasoning:       str = ""

    @field_validator('sections')
    @classmethod
    def sections_not_empty(cls, v):
        if len(v) == 0: raise ValueError('sections ต้องมีอย่างน้อย 1 มาตรา')
        return v

def validate_llm_output(llm_dict: dict) -> tuple:
    try:
        parsed = LLMOutput(**llm_dict)
        return True, parsed, ''
    except Exception as e:
        return False, None, str(e)

def is_valid_json_output(result: dict) -> bool:
    required = ['case_id', 'verdict_mode', 'adjusted_confidence']
    return all(k in result for k in required) and 'error' not in result

def compute_f1_sections(pred_sections: list, gold_sections: list) -> dict:
    pred_set = set(str(s) for s in pred_sections)
    gold_set = set(str(s) for s in gold_sections)
    if not pred_set and not gold_set: return {'precision': 1.0, 'recall': 1.0, 'f1': 1.0}
    if not pred_set or not gold_set:  return {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
    tp        = len(pred_set & gold_set)
    precision = tp / len(pred_set)
    recall    = tp / len(gold_set)
    f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    return {'precision': round(precision, 4), 'recall': round(recall, 4), 'f1': round(f1, 4)}

print("✅ [Step 8] Validation Layer พร้อม (Legal Syllogism Schema)")


✅ [Step 8] Validation Layer พร้อม (Legal Syllogism Schema)


## 🔬 Phase 8: การทดลองค้นหาและพิสูจน์กลยุทธ์ (Dev Set Experiments & Ablation)
### บันทึกประวัติการทดลองเพื่อหาจุดสมดุลที่ดีที่สุด (Optimization Timeline):
1. **Experiment 1 (Tournament 4 กลยุทธ์)**: 
   - Strategy 3 (Cluster-15: Neighbor +-2) ชนะขาดลอยด้วย F1: 0.4292
2. **Experiment 2 (Plan-3 Micro-Ablation Study ชำแหละ 5 ชิ้นส่วน)**:
   - 3A. Base-NoGate (F1: 0.4457)
   - 3B. Current-Base (F1: 0.4416)
   - 3C. Narrow +-1 (F1: 0.4202)
   - **3D. Wide-Cluster (+-3, Top-15, Gate ON) ชนะเลิศด้วย F1: 0.4542** 🏆
   - 3E. Deep-Context (Top-18) (F1: 0.4167)
*(เซลล์ด้านล่างนี้รันยืนยันผลลัพธ์ของแชมป์เปี้ยน 3D บน Dev Set 81 คดี)*


In [9]:
# ===============================================================
# [Step 9] DEV EVALUATION (CHAMPION CONFIG: 3D. Wide-Cluster)
# ===============================================================
import pandas as pd
import concurrent.futures
import time

CHAMPION_STRATEGY = {
    "name": "3D. Wide-Cluster (Expand+-3, Top-15, Gate ON)",
    "top_k": 20,
    "use_rerank": True,
    "rerank_n": 15,
    "anti_dangling": True,
    "expand": 3
}

print(f"🎯 Evaluating Champion Strategy on Dev Set (81 คดี): {CHAMPION_STRATEGY['name']}...")

def worker_eval(args):
    i, row = args
    cid = f"DEV-{i:04d}"
    fact, gold = str(row["fact"]), row["clean_sections"]
    res = analyze_criminal_liability_custom(fact, cid, CHAMPION_STRATEGY)
    scores = compute_f1_sections(res.get("sections", []), gold)
    return {**res, **scores, "gold_sections": gold, "fact": fact}

t0 = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    dev_results = list(executor.map(worker_eval, df_dev.iterrows()))
elapsed = time.time() - t0

valid = [r for r in dev_results if is_valid_json_output(r)]
p = sum(r["precision"] for r in valid) / max(len(valid), 1)
r = sum(r["recall"] for r in valid) / max(len(valid), 1)
f1 = sum(r["f1"] for r in valid) / max(len(valid), 1)

print("\n" + "=" * 60)
print("🏆 DEV SET EVALUATION RESULT (CHAMPION 3D)")
print("=" * 60)
print(f"  Precision   : {p:.4f}")
print(f"  Recall      : {r:.4f}")
print(f"  F1-Score    : {f1:.4f}")
print(f"  Time Taken  : {elapsed:.1f}s")
print("=" * 60)

with open('/kaggle/working/dev_results.json', 'w', encoding='utf-8') as f:
    json.dump(dev_results, f, ensure_ascii=False, indent=2)
print("💾 บันทึกผลลัพธ์ /kaggle/working/dev_results.json เรียบร้อย")


🎯 Evaluating Champion Strategy on Dev Set (81 คดี): 3D. Wide-Cluster (Expand+-3, Top-15, Gate ON)...

🏆 DEV SET EVALUATION RESULT (CHAMPION 3D)
  Precision   : 0.5247
  Recall      : 0.4300
  F1-Score    : 0.4490
  Time Taken  : 114.9s
💾 บันทึกผลลัพธ์ /kaggle/working/dev_results.json เรียบร้อย


## 📊 Phase 9: การวิเคราะห์ข้อผิดพลาดเชิงลึก (Error Distribution Analysis)
จำแนกประเภทข้อผิดพลาดของคำทำนาย (CORRECT, INCOMPLETE, WRONG_SECTION, HALLUCINATION, NO_OUTPUT)
เพื่อค้นหาจุดคอขวดที่สามารถดึงคะแนนกลับคืนมาได้


In [10]:
def categorize_error(result: dict, gold_sections: list) -> str:
    if 'error' in result: return 'NO_OUTPUT'
    pred_secs = set(result.get('sections', []))
    gold_secs = set(str(s) for s in gold_sections)
    invalid   = set(result.get('invalid_sections_flag', []))
    if pred_secs == gold_secs:              return 'CORRECT'
    if invalid:                             return 'HALLUCINATION'
    if pred_secs and not (pred_secs & gold_secs): return 'WRONG_SECTION'
    if gold_secs - pred_secs:               return 'INCOMPLETE'
    return 'PARTIAL'


error_counts   = {'CORRECT':0,'WRONG_SECTION':0,'HALLUCINATION':0,'INCOMPLETE':0,'PARTIAL':0,'NO_OUTPUT':0}
error_examples = {k: [] for k in error_counts}

for result in dev_results:
    gold = result.get('gold_sections', [])
    cat  = categorize_error(result, gold)
    error_counts[cat] += 1
    if cat != 'CORRECT' and len(error_examples[cat]) < 3:
        error_examples[cat].append({'case_id': result.get('case_id'), 'gold': gold,
                                    'predicted': result.get('sections', []), 'confidence': result.get('adjusted_confidence')})

total = len(dev_results)
print('=' * 55)
print('🔍 ERROR ANALYSIS')
print('=' * 55)
for cat, count in sorted(error_counts.items(), key=lambda x: -x[1]):
    bar = '█' * int(count / total * 30)
    print(f"  {cat:<15}: {count:>3} ({count/total:.1%}) {bar}")

with open('/kaggle/working/error_analysis.json', 'w', encoding='utf-8') as f:
    json.dump({'error_counts': error_counts, 'error_examples': error_examples}, f, ensure_ascii=False, indent=2)
print('\n✅ [Step 10] Error Analysis เสร็จสิ้น')

🔍 ERROR ANALYSIS
  INCOMPLETE     :  33 (40.7%) ████████████
  WRONG_SECTION  :  26 (32.1%) █████████
  CORRECT        :  18 (22.2%) ██████
  PARTIAL        :   4 (4.9%) █
  HALLUCINATION  :   0 (0.0%) 
  NO_OUTPUT      :   0 (0.0%) 

✅ [Step 10] Error Analysis เสร็จสิ้น


## 🤝 Phase 10: ระบบตรวจจับมาตราคู่หู (Co-occurrence Table Construction)
ขุดสถิติความถี่ของการฟ้องร่วมกันของแต่ละมาตรา (Co-occurrence Patterns) จากประวัติคำพิพากษาในอดีต 
สร้างเป็นตารางกฎหมายคู่หูอัตโนมัติ (Companion Suggestions & Pruning Rules)


In [11]:
def parse_lawids_for_cooc(lawids_str: str) -> list:
    if not isinstance(lawids_str, str) or not lawids_str.strip(): return []
    sections = []
    for token in lawids_str.split(','):
        m = re.match(r'CC-0*(\d+\S*)-\S+', token.strip())
        if m: sections.append(m.group(1))
    return sections

def build_cooccurrence(df, lawids_col: str = 'lawids') -> tuple:
    section_counts = defaultdict(int)
    pair_counts    = defaultdict(lambda: defaultdict(int))
    for lawids_str in df[lawids_col]:
        secs = list(set(parse_lawids_for_cooc(lawids_str)))
        for s in secs: section_counts[s] += 1
        for a, b in combinations(secs, 2):
            pair_counts[a][b] += 1
            pair_counts[b][a] += 1
    cooc = {}
    for a, partners in pair_counts.items():
        cooc[a] = {b: {'count': cnt, 'prob': round(cnt/section_counts[a], 3)} for b, cnt in partners.items()}
    return cooc, dict(section_counts)

DEPENDENT_SECTIONS = {'83','84','86','87','68','1','59','60','61'}

def build_companion_table(cooc: dict, section_counts: dict, threshold: float = 0.55) -> dict:
    table = {}
    for section, partners in cooc.items():
        companions = sorted([b for b, info in partners.items() if info['prob'] >= threshold],
                            key=lambda b: -partners[b]['prob'])
        table[section] = {'suggest_if_missing': companions, 'freq': section_counts.get(section, 0),
                          'top_pairs': {b: partners[b]['prob'] for b in companions}}
    return table

def apply_companion_check(result: dict, companion_table: dict, penalty_missing: int = 8, penalty_dangling: int = 5) -> dict:
    result = copy.deepcopy(result)
    if 'error' in result: return result
    pred_secs = {str(s) for s in result.get('sections', [])}
    warnings, suggestions, dangling = [], set(), set()
    for sec in pred_secs:
        if sec not in DEPENDENT_SECTIONS and sec in companion_table:
            for companion in companion_table[sec].get('suggest_if_missing', []):
                if companion not in pred_secs:
                    prob = companion_table[sec]['top_pairs'].get(companion, 0)
                    warnings.append(f'⚠️  ม.{sec} มักมากับ ม.{companion} ({prob:.0%}) แต่ไม่พบ')
                    suggestions.add(companion)
    non_dep = pred_secs - DEPENDENT_SECTIONS
    for sec in pred_secs:
        if sec in DEPENDENT_SECTIONS and not non_dep:
            dangling.add(sec)
            warnings.append(f'🔴 ม.{sec} ไม่มีมาตราหลัก')
    current_conf = result.get('adjusted_confidence') or 80
    if isinstance(current_conf, str): current_conf = int(re.sub(r'[^\d]', '', current_conf) or 80)
    penalty  = penalty_missing * min(len(suggestions), 3) + penalty_dangling * len(dangling)
    new_conf = max(0, int(current_conf) - penalty)
    result.update({'companion_warnings': warnings, 'companion_suggestions': list(suggestions),
                   'dangling_sections': list(dangling), 'adjusted_confidence': new_conf,
                   'possible_sections': list(set(result.get('possible_sections', [])) | suggestions - pred_secs)})
    return result

cooc, section_counts = build_cooccurrence(df_judgement, lawids_col='lawids')
COMPANION_TABLE = build_companion_table(cooc, section_counts, threshold=0.55)

dev_results_v2 = []
sug_count = dan_count = 0
for result in dev_results:
    r2 = apply_companion_check(result, COMPANION_TABLE)
    dev_results_v2.append(r2)
    if r2.get('companion_suggestions'): sug_count += 1
    if r2.get('dangling_sections'):     dan_count += 1

print(f'\n✅ [Step 11] Companion Table พร้อม ({len(COMPANION_TABLE)} มาตรา)')
print(f'   SUGGEST triggered: {sug_count}/{len(dev_results)} ({sug_count/len(dev_results):.0%})')
print(f'   PRUNE triggered  : {dan_count}/{len(dev_results)} ({dan_count/len(dev_results):.0%})')

with open('/kaggle/working/companion_table.json', 'w', encoding='utf-8') as f:
    json.dump(COMPANION_TABLE, f, ensure_ascii=False, indent=2)



✅ [Step 11] Companion Table พร้อม (61 มาตรา)
   SUGGEST triggered: 8/81 (10%)
   PRUNE triggered  : 1/81 (1%)


## ⚡ Phase 11: การตรวจสอบคู่หูแบบคู่ขนาน (Parallel Fast-Verification)
นำคดีที่มีข้อสงสัยว่าขาดมาตราคู่หู ส่งเข้ากระบวนการ Fast Verification ด้วย Multi-threading
โดยถาม LLM สั้นๆ แบบ Binary (True/False) ใช้เวลาเพียงไม่กี่วินาที ช่วยดึงคะแนน Recall เพิ่มขึ้นทันที


In [12]:
def build_verify_prompt(fact: str, current_sections: list, candidate: str) -> tuple:
    current_str = ', '.join(f'ม.{s}' for s in current_sections)
    sys_p = """คุณคือผู้เชี่ยวชาญกฎหมายอาญาไทย ตอบเฉพาะ JSON เท่านั้น
รูปแบบ: {"add": true/false, "reason": "เหตุผลสั้นๆ"}
- add: true  = ข้อเท็จจริงรองรับมาตรานี้ชัดเจน
- add: false = ไม่รองรับ"""
    usr_p = f"ข้อเท็จจริง: {fact}\n\nมาตราที่ระบุแล้ว: {current_str}\nมาตราที่พิจารณาเพิ่ม: ม.{candidate}\n\nข้อเท็จจริงรองรับ ม.{candidate} หรือไม่?"
    return sys_p, usr_p

def verify_single_candidate(args):
    case_id, fact, current_sec, candidate = args
    sys_p, usr_p = build_verify_prompt(fact, current_sec, candidate)
    raw = call_typhoon(sys_p, usr_p)
    parsed = parse_llm_json(raw) if raw else None
    if parsed and parsed.get('add', False) and str(candidate) in list_of_all_legal_sections:
        return case_id, str(candidate), True
    return case_id, str(candidate), False

def sequential_verify(cases_to_verify: list, results_ref: list, index_ref: dict, label: str = '', max_workers: int = 10) -> tuple:
    """Parallel Companion Verify ด้วย Multi-threading ลดเวลาจากหลายนาทีเหลือไม่กี่วินาที"""
    results_v3 = copy.deepcopy(results_ref)
    tasks = []
    
    for result in cases_to_verify:
        case_id     = result.get('case_id')
        fact        = result.get('fact', '')
        current_sec = result.get('sections', [])
        suggestions = result.get('companion_suggestions', [])

        if not fact and case_id:
            try:
                idx = int(case_id.replace('DEV-', '').replace('TEST-', ''))
                src_df = df_dev if 'DEV-' in case_id else df_test
                if idx < len(src_df):
                    fact = str(src_df.iloc[idx]['fact'])
            except Exception:
                pass
        for cand in suggestions:
            tasks.append((case_id, fact, current_sec, cand))

    added_map = defaultdict(list)
    added = rejected = 0

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        for cid, cand, is_add in tqdm(executor.map(verify_single_candidate, tasks), total=len(tasks), desc=f"Verify {label}"):
            if is_add:
                added_map[cid].append(cand)
                added += 1
            else:
                rejected += 1

    for cid, new_secs in added_map.items():
        if cid in index_ref:
            idx_r = index_ref[cid]
            combined = list(set(results_v3[idx_r].get('sections', [])) | set(new_secs))
            gold     = results_v3[idx_r].get('gold_sections', [])
            results_v3[idx_r]['sections'] = combined
            results_v3[idx_r]['verified_additions'] = new_secs
            results_v3[idx_r].update(compute_f1_sections(combined, gold))

    return results_v3, added, rejected

cases_to_verify = [r for r in dev_results_v2 if r.get('companion_suggestions') and 'error' not in r]
print(f"\n✅ [Step 12] Companion Verify: {len(cases_to_verify)} คดี (Parallel Fast-Verify)")

v3_index = {r.get('case_id'): i for i, r in enumerate(dev_results_v2)}
dev_results_v3, verified_added, verified_rejected = sequential_verify(
    cases_to_verify, dev_results_v2, v3_index, label='DEV'
)

print(f"   เพิ่มมาตรา: {verified_added} | ปฏิเสธ: {verified_rejected}")
f1_v2 = [compute_f1_sections(r.get('sections',[]), r.get('gold_sections',[]))['f1'] for r in dev_results_v2]
f1_v3 = [r.get('f1', 0.0) for r in dev_results_v3]
print(f"   Dev F1 After Verify: {sum(f1_v3)/len(f1_v3):.4f} (Before: {sum(f1_v2)/len(f1_v2):.4f})")

with open('/kaggle/working/dev_results_v3.json', 'w', encoding='utf-8') as f:
    json.dump(dev_results_v3, f, ensure_ascii=False, indent=2)
print('✅ [Step 12] Companion Verify เสร็จสิ้น')



✅ [Step 12] Companion Verify: 8 คดี (Parallel Fast-Verify)


Verify DEV: 100%|██████████| 11/11 [00:03<00:00,  3.51it/s]

   เพิ่มมาตรา: 8 | ปฏิเสธ: 3
   Dev F1 After Verify: 0.4643 (Before: 0.4490)
✅ [Step 12] Companion Verify เสร็จสิ้น


## 🏁 Phase 12: การทดสอบเต็มรูปแบบ 965 คดีจริง (Full Test Benchmark & Auto-Recovery)
ประเมินผลระบบบนชุดทดสอบจริง TSCC Test Set (965 คดี):
* **Parallel Throughput**: 25 Concurrent Workers (ยิงได้เต็มเพดาน 140-160 RPM โดยไม่โดน Rate Limit)
* **3-Stage Auto-Recovery**: ระบบกู้ชีพอัตโนมัติ (ขยาย K=50, Temp=0.6) เมื่อพบคำตอบว่าง
* **Automatic Checkpointing**: บันทึกผลทุก 50 คดี ป้องกันข้อผิดพลาดระหว่างรัน
* **Final Companion Check**: ตรวจสอบมาตราคู่หูรอบสุดท้ายก่อนสรุปคะแนนทางการ


In [13]:
# --- [0] CHAMPION STRATEGY CONFIGURATION ---
if 'CHAMPION_STRATEGY' not in globals() or CHAMPION_STRATEGY is None:
    CHAMPION_STRATEGY = {
        'name': '3D. Wide-Cluster (Expand+-3, Top-15, Gate ON)',
        'top_k': 20,
        'use_rerank': True,
        'rerank_n': 15,
        'anti_dangling': True,
        'expand': 3
    }
print(f"🎯 Using Champion Strategy: {CHAMPION_STRATEGY['name']}")

SLEEP_BETWEEN = 0.0
import threading
import concurrent.futures


CHECKPOINT_PATH  = '/kaggle/working/test_checkpoint.json'
CHECKPOINT_EVERY = 50
MAX_WORKERS      = 25  
_lock            = threading.Lock() 

# --- [1] Logic การกู้คืนอัตโนมัติ (Recovery Logic) ---
def analyze_recovery_v1(fact: str, case_id: str, gold_secs: list) -> tuple:
    """
    Recovery pipeline — รันเมื่อ main pipeline ล้มเหลว
    ลองทีละ stage จนกว่าจะได้ sections
    คืนค่า (result, f1_score, stage_name)
    """
    stages = [
        ("K50_T02", {"top_k": 50, "temperature": 0.2}),
        ("K50_T06", {"top_k": 50, "temperature": 0.6}),
        ("K25_T05", {"top_k": 25, "temperature": 0.5}),
    ]
    for stage_name, params in stages:
        try:
            temp     = params.pop("temperature", 0.3)
            top_k    = params.get("top_k", 25)
            retrieved = hybrid_retrieve(fact, top_k=top_k)
            sys_p, usr_p = build_prompt(fact, retrieved, case_id)
            usr_p += "\n\n**คำสั่งพิเศษ: ต้องระบุมาตราอย่างน้อย 1 ข้อ ห้ามตอบ sections:[] เด็ดขาด**"
            response = typhoon_client.chat.completions.create(
                model=TYPHOON_MODEL,
                messages=[
                    {"role": "system", "content": sys_p},
                    {"role": "user",   "content": usr_p},
                ],
                max_tokens=MAX_TOKENS,
                temperature=temp,
                response_format={"type": "json_object"},
            )
            llm_dict = parse_llm_json(response.choices[0].message.content)
            if llm_dict and llm_dict.get("sections"):
                result = post_process_llm_output(
                    llm_output=llm_dict,
                    legal_db=list_of_all_legal_sections,
                    context_section_ids=retrieved["context_section_ids"],
                    fact=fact,
                )
                f1 = compute_f1_sections(result.get("sections", []), gold_secs)["f1"]
                return result, f1, stage_name
        except Exception as e:
            print(f"   ❌ Recovery {stage_name} failed: {e}")
        time.sleep(1)

    return {"case_id": case_id, "sections": [], "error": "Recovery Failed"}, 0.0, "FAILED"

def analyze_with_auto_recovery(fact, case_id, gold_secs):
    """
    Pipeline หลัก: ลองรันปกติ (K=25) ถ้าล้มเหลวให้เข้าลูปกู้คืนทันที (K=50, Temp=0.6)[cite: 2]
    """
    # รอบที่ 1: รันแบบปกติ (Fast Track)[cite: 2]
    result = analyze_criminal_liability_custom(fact, case_id, strat=CHAMPION_STRATEGY) 
    
    # รอบที่ 2: ตรวจสอบปัญหา (Empty Sections หรือ LLM Error)[cite: 2]
    if not result.get("sections") or "error" in result:
        print(f"   🆘 {case_id} พบปัญหา (Empty/Error) -> กำลังเริ่ม Sequential Recovery...")
    
        recovered_res, new_f1, stage = analyze_recovery_v1(fact, case_id, gold_secs)
        
        if recovered_res.get("sections"):
            result = recovered_res
            result['recovery_log'] = f"Recovered by {stage}"
            print(f"   ✅ {case_id} กู้คืนสำเร็จ! (F1: {new_f1:.4f})")
            
    return result

# --- [2] Worker สำหรับการรันแบบ Parallel ---

def _run_one_case(args):
    """Worker ที่เรียกใช้ระบบ Auto-Recovery[cite: 2]"""
    i, row    = args
    case_id   = f'TEST-{i:04d}'
    fact      = str(row['fact'])
    gold_secs = row['clean_sections']
    
    # เรียกผ่านระบบกู้คืนอัตโนมัติ
    result = analyze_with_auto_recovery(fact, case_id, gold_secs)
    
    # แนบข้อมูลพื้นฐานกลับไปเพื่อใช้คำนวณ Metrics[cite: 2]
    result['gold_sections'] = gold_secs
    result['fact']          = fact
    
    if is_valid_json_output(result):
        result.update(compute_f1_sections(result.get('sections', []), gold_secs))
    else:
        result.update({'precision': 0.0, 'recall': 0.0, 'f1': 0.0})
        
    time.sleep(SLEEP_BETWEEN)
    return result

# --- [3] ส่วนควบคุมการรัน (Main Execution) ---

if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH, 'r', encoding='utf-8') as f:
        test_results = json.load(f)
    start_idx = len(test_results)
    print(f'♻️ Resume จาก checkpoint: {start_idx}/{len(df_test)} คดี')
else:
    test_results = []
    start_idx    = 0
    print('🆕 เริ่มใหม่ตั้งแต่ต้น')

print(f'\n⚠️ เริ่มรัน Full Run 965 คดี (ระบบ Auto-Recovery เปิดใช้งาน)\n')

remaining_rows = [(i, row) for i, row in df_test.iterrows() if i >= start_idx]
start_time = time.time()

with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(_run_one_case, args): args[0] for args in remaining_rows}
    
    for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc='Test Eval'):
        result = future.result()
        with _lock:
            test_results.append(result)
            if len(test_results) % CHECKPOINT_EVERY == 0:
                with open(CHECKPOINT_PATH, 'w', encoding='utf-8') as f:
                    json.dump(test_results, f, ensure_ascii=False)
                print(f'💾 Checkpoint: {len(test_results)}/{len(df_test)} คดี')

elapsed = time.time() - start_time
print(f'\n✅ รัน pipeline ครบ {len(test_results)} คดี | เวลา: {elapsed:.1f}s')

# --- [4] Companion Check & Final Statistics ---[cite: 2]

print('\n⏳ เริ่มขั้นตอนสุดท้าย: Apply Companion Check & Verify...')
test_results_v2 = [apply_companion_check(r, COMPANION_TABLE) for r in test_results]
cases_tv        = [r for r in test_results_v2 if r.get('companion_suggestions') and 'error' not in r]

tv3_index = {r.get('case_id'): i for i, r in enumerate(test_results_v2)}
test_results_v3, tv_added, tv_rejected = sequential_verify(
    cases_tv, test_results_v2, tv3_index, label='TEST'
)

valid_results = [r for r in test_results_v3 if 'error' not in r and 'f1' in r]
n_total       = len(test_results_v3)
n_valid       = len(valid_results)
avg_f1        = sum(r['f1'] for r in valid_results) / n_valid if n_valid else 0
json_validity = (n_valid / n_total) * 100

print('\n' + '=' * 55)
print('🏁 FINAL TEST SET RESULTS (AUTO-RECOVERY ENABLED)')
print('=' * 55)
print(f"  Total Cases   : {n_total}")
print(f"  JSON Validity : {json_validity:.1f}% ({n_valid}/{n_total})")
print(f"  Avg F1 Score  : {avg_f1:.4f}")
print(f"  Total Time    : {elapsed:.1f}s")
print('=' * 55)

output = {
    'timestamp': datetime.now().isoformat(),
    'results': test_results_v3,
    'summary': {'avg_f1': round(avg_f1, 4), 'validity': round(json_validity, 2)}
}
with open('/kaggle/working/test_results_final.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

if os.path.exists(CHECKPOINT_PATH):
    os.remove(CHECKPOINT_PATH)
    print('🗑️ ลบ checkpoint แล้ว')

print('💾 บันทึกผลสมบูรณ์ที่: /kaggle/working/test_results_final.json')


🎯 Using Champion Strategy: 3D. Wide-Cluster (Expand+-3, Top-15, Gate ON)
🆕 เริ่มใหม่ตั้งแต่ต้น

⚠️ เริ่มรัน Full Run 965 คดี (ระบบ Auto-Recovery เปิดใช้งาน)



Test Eval:   5%|▌         | 50/965 [01:21<13:50,  1.10it/s]

💾 Checkpoint: 50/965 คดี
   🆘 TEST-0045 พบปัญหา (Empty/Error) -> กำลังเริ่ม Sequential Recovery...


Test Eval:   7%|▋         | 63/965 [01:34<14:18,  1.05it/s]

   🆘 TEST-0057 พบปัญหา (Empty/Error) -> กำลังเริ่ม Sequential Recovery...


...

   ✅ TEST-0964 กู้คืนสำเร็จ! (F1: 0.0000)


Test Eval: 100%|██████████| 965/965 [19:59<00:00,  1.24s/it]



✅ รัน pipeline ครบ 965 คดี | เวลา: 1199.7s

⏳ เริ่มขั้นตอนสุดท้าย: Apply Companion Check & Verify...


Verify TEST: 100%|██████████| 130/130 [00:17<00:00,  7.50it/s]


🏁 FINAL TEST SET RESULTS (AUTO-RECOVERY ENABLED)
  Total Cases   : 965
  JSON Validity : 100.0% (965/965)
  Avg F1 Score  : 0.4499
  Total Time    : 1199.7s
🗑️ ลบ checkpoint แล้ว
💾 บันทึกผลสมบูรณ์ที่: /kaggle/working/test_results_final.json


## 🔍 Phase 13: วิเคราะห์เคสตกหล่น (Zero-F1 Deep Inspection & Production Insights)
เจาะลึก 386 คดีที่ได้ F1 = 0.0 ตามเกณฑ์ Strict Exact Match:
* วิเคราะห์ความต่างระหว่างข้อหาธรรมดา vs ข้อหาฉกรรจ์ (ม.288 vs ม.289)
* วิเคราะห์เส้นแบ่งดูหมิ่น vs หมิ่นประมาท (ม.393 vs ม.328)
* สรุปข้อแนะนำในการออกแบบ UI แสดงผลบน Production (Hugging Face Spaces)


In [14]:
# Appendix A: ดู System Errors และ Zero-F1
with open('/kaggle/working/test_results_final.json', 'r', encoding='utf-8') as f:
    final_data = json.load(f)

all_results   = final_data['results']
system_errors = [r for r in all_results if 'error' in r]
zero_f1_cases = [r for r in all_results if 'error' not in r and r.get('f1') == 0]
empty_pred    = [r for r in all_results if not r.get('sections') and r.get('gold_sections')]

print('📊 สรุปภาพรวมปัญหา:')
print(f"   System Errors (LLM Failure) : {len(system_errors)} คดี")
print(f"   Zero F1 (ตอบผิดทั้งหมด)   : {len(zero_f1_cases)} คดี")
print(f"   Empty Prediction (ไม่ตอบ)  : {len(empty_pred)} คดี")
print('=' * 50)

for i, case in enumerate(system_errors[:5]):
    print(f"❌ [{i+1}] {case.get('case_id')} | สาเหตุ: {case.get('error')}")

if zero_f1_cases:
    print('\n💡 ตัวอย่าง Zero F1 (3 คดี):')
    for case in random.sample(zero_f1_cases, min(3, len(zero_f1_cases))):
        print(f"📌 {case.get('case_id')} | Gold: {case.get('gold_sections')} | Pred: {case.get('sections')}")

📊 สรุปภาพรวมปัญหา:
   System Errors (LLM Failure) : 0 คดี
   Zero F1 (ตอบผิดทั้งหมด)   : 380 คดี
   Empty Prediction (ไม่ตอบ)  : 6 คดี

💡 ตัวอย่าง Zero F1 (3 คดี):
📌 TEST-0751 | Gold: ['335'] | Pred: ['334']
📌 TEST-0560 | Gold: ['334', '1'] | Pred: ['83', '353']
📌 TEST-0819 | Gold: ['329', '326'] | Pred: ['328']
